# Automated Contradiction Detection in News Articles
**Joel Adu-Kwarteng & Kishore Vasudevan**  
NLP Final Project — Spring 2026

---

This notebook trains and evaluates three models on the SNLI dataset for 3-class NLI:
1. **TF-IDF + Logistic Regression** (baseline)
2. **TF-IDF + Linear SVM** (baseline)
3. **BERT (bert-base-uncased)** (fine-tuned transformer)

All results, metrics, and figures are saved to Google Drive.

## 0. Setup

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install -q datasets transformers[torch] scikit-learn seaborn

In [ ]:
# Verify GPU availability
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack, csr_matrix

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────

# Paths (Google Drive)
DRIVE_BASE = "/content/drive/MyDrive/NLP_Project"
RESULTS_DIR = os.path.join(DRIVE_BASE, "results")
MODELS_DIR = os.path.join(DRIVE_BASE, "models")
FIGURES_DIR = os.path.join(DRIVE_BASE, "figures")

for d in [RESULTS_DIR, MODELS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# Labels
LABEL_NAMES = ["entailment", "neutral", "contradiction"]
NUM_LABELS = 3
SEED = 42

# TF-IDF
TFIDF_MAX_FEATURES = 50_000
TFIDF_NGRAM_RANGE = (1, 2)

# BERT
BERT_MODEL_NAME = "bert-base-uncased"
BERT_MAX_LENGTH = 128
BERT_BATCH_SIZE_TRAIN = 32
BERT_BATCH_SIZE_EVAL = 64
BERT_LR = 2e-5
BERT_EPOCHS = 3

print(f"Output directory: {DRIVE_BASE}")

## 1. Load and Preprocess SNLI Dataset

In [ ]:
# Load SNLI
dataset = load_dataset("stanfordnlp/snli")

# Filter out examples with no gold label (label == -1)
for split in ["train", "validation", "test"]:
    original = len(dataset[split])
    dataset[split] = dataset[split].filter(lambda x: x["label"] != -1)
    filtered = len(dataset[split])
    print(f"{split}: {original} → {filtered} ({original - filtered} removed)")

In [ ]:
# Explore the data
print("\nSample examples:")
print("-" * 80)
for i in range(5):
    ex = dataset["train"][i]
    print(f"Premise:    {ex['premise']}")
    print(f"Hypothesis: {ex['hypothesis']}")
    print(f"Label:      {LABEL_NAMES[ex['label']]}")
    print("-" * 80)

In [ ]:
# Check class distribution
for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    counts = pd.Series(labels).value_counts().sort_index()
    total = len(labels)
    print(f"\n{split} set ({total} examples):")
    for idx, count in counts.items():
        print(f"  {LABEL_NAMES[idx]}: {count} ({count/total*100:.1f}%)")

In [ ]:
# Extract texts and labels for baselines
train_premises = dataset["train"]["premise"]
train_hypotheses = dataset["train"]["hypothesis"]
y_train = np.array(dataset["train"]["label"])

val_premises = dataset["validation"]["premise"]
val_hypotheses = dataset["validation"]["hypothesis"]
y_val = np.array(dataset["validation"]["label"])

test_premises = dataset["test"]["premise"]
test_hypotheses = dataset["test"]["hypothesis"]
y_test = np.array(dataset["test"]["label"])

print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

## 2. TF-IDF Feature Engineering

We represent each premise-hypothesis pair as:

$$\mathbf{x} = [P;\ H;\ |P - H|;\ P \odot H]$$

Where $P$ and $H$ are TF-IDF vectors. The difference features capture what's unique to each sentence (useful for contradictions), and the product features capture shared content (useful for entailment).

In [ ]:
# Fit TF-IDF on all training text
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    sublinear_tf=True,
    dtype=np.float32,
)

all_train_text = train_premises + train_hypotheses
vectorizer.fit(all_train_text)
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

In [ ]:
def make_interaction_features(premises, hypotheses, vectorizer, split_name=""):
    """
    Build TF-IDF interaction features for sentence pairs.
    Returns sparse matrix: [P; H; |P-H|; P*H]
    """
    P = vectorizer.transform(premises)
    H = vectorizer.transform(hypotheses)

    diff = csr_matrix(np.abs(P - H))    # what differs
    prod = P.multiply(H)                  # what overlaps

    X = hstack([P, H, diff, prod], format="csr")
    print(f"{split_name} features: {X.shape} (nnz: {X.nnz:,})")
    return X

X_train = make_interaction_features(train_premises, train_hypotheses, vectorizer, "Train")
X_val = make_interaction_features(val_premises, val_hypotheses, vectorizer, "Val")
X_test = make_interaction_features(test_premises, test_hypotheses, vectorizer, "Test")

## 3. Evaluation Helper Functions

In [ ]:
def evaluate_model(y_true, y_pred, model_name, model_key):
    """
    Full evaluation: print report, compute metrics, save everything.
    Returns metrics dict.
    """
    # Print classification report
    print(f"\n{'='*60}")
    print(f"Results: {model_name}")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

    # Compute metrics dict
    metrics = {
        "model": model_name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro")),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro")),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }

    # Per-class metrics
    per_p = precision_score(y_true, y_pred, average=None)
    per_r = recall_score(y_true, y_pred, average=None)
    per_f = f1_score(y_true, y_pred, average=None)
    for i, name in enumerate(LABEL_NAMES):
        metrics[f"{name}_precision"] = float(per_p[i])
        metrics[f"{name}_recall"] = float(per_r[i])
        metrics[f"{name}_f1"] = float(per_f[i])

    # Save metrics JSON
    json_path = os.path.join(RESULTS_DIR, f"{model_key}_metrics.json")
    with open(json_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved: {json_path}")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    # Save CM as CSV
    cm_df = pd.DataFrame(cm, index=LABEL_NAMES, columns=LABEL_NAMES)
    cm_df.index.name = "True"
    cm_df.columns.name = "Predicted"
    csv_path = os.path.join(RESULTS_DIR, f"{model_key}_confusion_matrix.csv")
    cm_df.to_csv(csv_path)

    # Plot CM
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
    axes[0].set_title(f"{model_name} — Counts")
    axes[0].set_ylabel("True Label")
    axes[0].set_xlabel("Predicted Label")

    sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues",
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[1])
    axes[1].set_title(f"{model_name} — Normalized")
    axes[1].set_ylabel("True Label")
    axes[1].set_xlabel("Predicted Label")

    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, f"{model_key}_confusion_matrix.png")
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Figure saved: {fig_path}")

    return metrics

## 4. Baseline 1: TF-IDF + Logistic Regression

In [ ]:
print("Training Logistic Regression...")
lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="lbfgs",
    multi_class="multinomial",
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)

start = time.time()
lr_model.fit(X_train, y_train)
lr_time = time.time() - start
print(f"Training time: {lr_time:.1f}s")

In [ ]:
# Evaluate Logistic Regression
lr_preds = lr_model.predict(X_test)
lr_metrics = evaluate_model(y_test, lr_preds, "TF-IDF + Logistic Regression", "tfidf_lr")
lr_metrics["train_time_s"] = lr_time

## 5. Baseline 2: TF-IDF + Linear SVM

In [ ]:
print("Training Linear SVM...")
svm_model = LinearSVC(
    max_iter=1000,
    C=1.0,
    random_state=SEED,
    verbose=1,
)

start = time.time()
svm_model.fit(X_train, y_train)
svm_time = time.time() - start
print(f"Training time: {svm_time:.1f}s")

In [ ]:
# Evaluate SVM
svm_preds = svm_model.predict(X_test)
svm_metrics = evaluate_model(y_test, svm_preds, "TF-IDF + Linear SVM", "tfidf_svm")
svm_metrics["train_time_s"] = svm_time

In [ ]:
# Save baseline models
import joblib
joblib.dump(vectorizer, os.path.join(MODELS_DIR, "tfidf_vectorizer.joblib"))
joblib.dump(lr_model, os.path.join(MODELS_DIR, "tfidf_lr.joblib"))
joblib.dump(svm_model, os.path.join(MODELS_DIR, "tfidf_svm.joblib"))
print("Baseline models saved to Drive.")

## 6. Fine-Tune BERT

We fine-tune `bert-base-uncased` using the HuggingFace Trainer API.
Premise and hypothesis are fed as a sentence pair, with BERT's `[SEP]` token separating them.

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

# Tokenize all splits
def tokenize_fn(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        max_length=BERT_MAX_LENGTH,
        padding="max_length",
    )

print("Tokenizing train split...")
train_tok = dataset["train"].map(tokenize_fn, batched=True, batch_size=1000)
print("Tokenizing validation split...")
val_tok = dataset["validation"].map(tokenize_fn, batched=True, batch_size=1000)
print("Tokenizing test split...")
test_tok = dataset["test"].map(tokenize_fn, batched=True, batch_size=1000)

# Set format for PyTorch
for ds in [train_tok, val_tok, test_tok]:
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenization complete!")

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=NUM_LABELS,
)

# Metrics callback for Trainer
def compute_metrics_trainer(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "macro_f1": macro_f1}

# Training arguments
training_args = TrainingArguments(
    output_dir=os.path.join(MODELS_DIR, "bert_checkpoints"),
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=BERT_BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=BERT_BATCH_SIZE_EVAL,
    learning_rate=BERT_LR,
    weight_decay=0.01,
    warmup_steps=500,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=500,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics_trainer,
)

print("Ready to train!")
print(f"  Train examples: {len(train_tok)}")
print(f"  Steps/epoch: {len(train_tok) // BERT_BATCH_SIZE_TRAIN}")
print(f"  Total steps: {(len(train_tok) // BERT_BATCH_SIZE_TRAIN) * BERT_EPOCHS}")

In [ ]:
# Train BERT (this takes ~45-60 min on free Colab T4)
start = time.time()
trainer.train()
bert_time = time.time() - start
print(f"\nBERT training time: {bert_time:.1f}s ({bert_time/60:.1f} min)")

In [ ]:
# Evaluate BERT on test set
test_output = trainer.predict(test_tok)
bert_preds = np.argmax(test_output.predictions, axis=-1)

bert_metrics = evaluate_model(y_test, bert_preds, "BERT (bert-base-uncased)", "bert")
bert_metrics["train_time_s"] = bert_time

In [ ]:
# Save BERT model
bert_save_path = os.path.join(MODELS_DIR, "bert_best")
trainer.save_model(bert_save_path)
tokenizer.save_pretrained(bert_save_path)
print(f"BERT model saved to: {bert_save_path}")

## 7. Model Comparison

In [ ]:
# Build comparison table
all_metrics = [lr_metrics, svm_metrics, bert_metrics]

rows = []
for m in all_metrics:
    rows.append({
        "Model": m["model"],
        "Accuracy": m["accuracy"],
        "Macro P": m["macro_precision"],
        "Macro R": m["macro_recall"],
        "Macro F1": m["macro_f1"],
        "Entail F1": m["entailment_f1"],
        "Neutral F1": m["neutral_f1"],
        "Contra F1": m["contradiction_f1"],
        "Train Time (s)": m.get("train_time_s", None),
    })

comparison_df = pd.DataFrame(rows)
numeric_cols = comparison_df.select_dtypes(include=[np.number]).columns
comparison_df[numeric_cols] = comparison_df[numeric_cols].round(4)

print("\n" + "=" * 90)
print("MODEL COMPARISON")
print("=" * 90)
print(comparison_df.to_string(index=False))

In [ ]:
# Save comparison table
csv_path = os.path.join(RESULTS_DIR, "model_comparison.csv")
comparison_df.to_csv(csv_path, index=False)
print(f"CSV saved: {csv_path}")

# Save as LaTeX table (paste directly into your paper!)
latex_path = os.path.join(RESULTS_DIR, "model_comparison.tex")
comparison_df.to_latex(latex_path, index=False, float_format="%.4f")
print(f"LaTeX table saved: {latex_path}")

# Print LaTeX for convenience
print("\n--- LaTeX Table (copy into your paper) ---")
print(comparison_df.to_latex(index=False, float_format="%.4f"))

In [ ]:
# Visual comparison — bar chart of F1 scores by class
fig, ax = plt.subplots(figsize=(10, 6))

model_names_short = ["TF-IDF + LR", "TF-IDF + SVM", "BERT"]
x = np.arange(len(LABEL_NAMES))
width = 0.25

for i, (m, name) in enumerate(zip(all_metrics, model_names_short)):
    f1_scores = [m[f"{lbl}_f1"] for lbl in LABEL_NAMES]
    bars = ax.bar(x + i * width, f1_scores, width, label=name)
    # Add value labels on bars
    for bar, score in zip(bars, f1_scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{score:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xlabel("Class")
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1 Score Comparison")
ax.set_xticks(x + width)
ax.set_xticklabels(LABEL_NAMES)
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "f1_comparison_by_class.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Comparison chart saved!")

## 8. Error Analysis

Let's look at examples the models get wrong, especially contradictions that were misclassified.

In [ ]:
# Analyze BERT errors on the contradiction class
contra_mask = y_test == 2  # contradiction label
bert_wrong_contra = (bert_preds != y_test) & contra_mask

wrong_indices = np.where(bert_wrong_contra)[0]
print(f"BERT misclassified {len(wrong_indices)} out of {contra_mask.sum()} contradictions")
print(f"Contradiction accuracy: {1 - len(wrong_indices)/contra_mask.sum():.4f}")

# Show some misclassified examples
print(f"\n--- Sample BERT errors on contradictions (showing up to 10) ---")
for idx in wrong_indices[:10]:
    print(f"\nExample {idx}:")
    print(f"  Premise:    {test_premises[idx]}")
    print(f"  Hypothesis: {test_hypotheses[idx]}")
    print(f"  True label: {LABEL_NAMES[y_test[idx]]}")
    print(f"  BERT pred:  {LABEL_NAMES[bert_preds[idx]]}")

In [ ]:
# Cross-model agreement analysis
all_correct = (lr_preds == y_test) & (svm_preds == y_test) & (bert_preds == y_test)
none_correct = (lr_preds != y_test) & (svm_preds != y_test) & (bert_preds != y_test)
bert_only = (bert_preds == y_test) & (lr_preds != y_test) & (svm_preds != y_test)

print("Cross-model agreement analysis:")
print(f"  All 3 correct:       {all_correct.sum()} ({all_correct.mean()*100:.1f}%)")
print(f"  None correct:        {none_correct.sum()} ({none_correct.mean()*100:.1f}%)")
print(f"  Only BERT correct:   {bert_only.sum()} ({bert_only.mean()*100:.1f}%)")

## 9. Summary

All results have been saved to Google Drive:
- **Metrics**: `NLP_Project/results/` (JSON + CSV + LaTeX)
- **Figures**: `NLP_Project/figures/` (confusion matrices, comparison charts)
- **Models**: `NLP_Project/models/` (sklearn + BERT checkpoints)

In [ ]:
# List all saved files
print("\n=== Saved Files ===")
for dirpath, dirnames, filenames in os.walk(DRIVE_BASE):
    level = dirpath.replace(DRIVE_BASE, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(dirpath)}/")
    subindent = " " * 2 * (level + 1)
    for file in sorted(filenames):
        filepath = os.path.join(dirpath, file)
        size = os.path.getsize(filepath)
        if size > 1e6:
            size_str = f"{size/1e6:.1f} MB"
        else:
            size_str = f"{size/1e3:.1f} KB"
        print(f"{subindent}{file} ({size_str})")